# Tool 9: Quote Optimization Workbench

## Purpose
The core strategy design tool for market making. This notebook lets you:
- See the orderbook as a depth snapshot at any timestep
- Simulate where your quotes would land inside the spread
- Find the optimal tradeoff between **edge** (profit per fill) and **fill probability**
- Track how inventory builds up under different aggressiveness settings
- Compare multiple aggressiveness levels side by side

## Why This Matters
Frankfurt Hedgehogs' entire Round 1 strategy was about this tradeoff:
> "Carefully optimizing the edge versus fill probability trade-off."

They placed passive orders that **overbid on bids and undercut on asks**
while maintaining positive edge, and flattened inventory at fair value
when it got too skewed. This tool lets you see and tune that exact behavior.

## Sub-Visualizations
- **9a**: Orderbook Depth Snapshot — see the book at any single timestep
- **9b**: Fill Probability vs Edge Curve — the key tradeoff plot
- **9c**: Inventory Trajectory Simulator — how position evolves over a day
- **9d**: Quote Competitiveness — would your quotes be best-in-book?
- **9e**: Aggressiveness Comparison Table — run multiple settings side by side

## How to Use
1. Start with the **Fill Probability vs Edge Curve** (9b) to find the
   optimal edge for each product
2. Use the **Depth Snapshot** (9a) to visualize the book and see where
   your quotes would land
3. Run the **Inventory Simulator** (9c) to check that your strategy
   doesn't hit position limits too often
4. Use the **Comparison Table** (9e) to pick your final parameters

## Dependencies
- `data_loader.py`, `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

PRODUCT = "ASH_COATED_OSMIUM"  # or "INTARIAN_PEPPER_ROOT"
DAY = -1

# --- For Depth Snapshot (9a) ---
# Which timestamp to show the orderbook snapshot for.
# Tip: start with 0, then try different timestamps to see how the book
# evolves. Use the orderbook viewer (Tool 1) to find interesting moments.
SNAPSHOT_TIMESTAMP = 5000

# Where your simulated quotes would be placed (offset from Wall Mid)
MY_BID_OFFSET = -2    # Quote bid at wall_mid + this (negative = below fair)
MY_ASK_OFFSET = 2     # Quote ask at wall_mid + this (positive = above fair)
MY_QUOTE_SIZE = 5     # How many lots to quote per side

# --- For Fill Probability Curve (9b) ---
# Range of edge values to test
EDGE_TEST_RANGE = range(1, 16)

# --- For Inventory Simulator (9c) ---
SIM_EDGE = 2             # Edge for the simulation
SIM_MAX_POSITION = 50    # Hard position limit
SIM_FLATTEN_THRESHOLD = 40  # Start flattening at this position
SIM_FLATTEN_EDGE = 0     # Edge at which to flatten (0 = at fair value)

# --- For Comparison Table (9e) ---
# List of (edge, max_position, flatten_threshold) tuples to compare
COMPARISON_CONFIGS = [
    {"label": "Conservative", "edge": 4, "max_pos": 50, "flatten": 40},
    {"label": "Moderate",     "edge": 3, "max_pos": 50, "flatten": 40},
    {"label": "Aggressive",   "edge": 2, "max_pos": 50, "flatten": 40},
    {"label": "Very Aggressive", "edge": 1, "max_pos": 50, "flatten": 40},
]

FIG_WIDTH = 18
FIG_HEIGHT = 6

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from data_loader import (
    load_prices, load_trades, compute_wall_mid, compute_raw_mid,
    filter_time_range, merge_trades_with_prices, PRODUCTS, AVAILABLE_DAYS,
)

prices = compute_wall_mid(compute_raw_mid(load_prices(day=DAY, product=PRODUCT)))
trades = load_trades(day=DAY, product=PRODUCT)
merged = merge_trades_with_prices(prices, trades)

print(f"Loaded {len(prices)} snapshots, {len(trades)} trades")
print(f"Product: {PRODUCT}, Day: {DAY}")

---
## 9a: Orderbook Depth Snapshot

Shows the full orderbook at a single timestep as a horizontal bar chart.
Your simulated quotes are overlaid as starred markers so you can see
exactly where they'd sit relative to existing liquidity.

**What to look for:**
- Are your quotes inside the spread (between best bid and best ask)?
- Are you overbidding the current best bid? (good — more fill probability)
- Are you behind the wall? (bad — you'll rarely get filled)
- How much edge do you have? (your quote price vs the Wall Mid)

In [ ]:
# ============================================================
# 9a: ORDERBOOK DEPTH SNAPSHOT
# ============================================================

# Find the row closest to the requested timestamp
snap_idx = (prices["timestamp"] - SNAPSHOT_TIMESTAMP).abs().idxmin()
snap = prices.loc[snap_idx]
wm = snap["wall_mid"]

fig, ax = plt.subplots(figsize=(12, FIG_HEIGHT))

# Collect bid levels
bid_levels = []
for level in [1, 2, 3]:
    p = snap[f"bid_price_{level}"]
    v = snap[f"bid_volume_{level}"]
    if not np.isnan(p):
        bid_levels.append((p, v))

# Collect ask levels
ask_levels = []
for level in [1, 2, 3]:
    p = snap[f"ask_price_{level}"]
    v = snap[f"ask_volume_{level}"]
    if not np.isnan(p):
        ask_levels.append((p, v))

# Plot bid bars (going left = negative volume)
for p, v in bid_levels:
    ax.barh(p, -v, height=0.8, color="tab:blue", alpha=0.6, edgecolor="black", linewidth=0.5)
    ax.text(-v - 0.5, p, f"{int(v)}@{int(p)}", va="center", ha="right", fontsize=9, color="tab:blue")

# Plot ask bars (going right = positive volume)
for p, v in ask_levels:
    ax.barh(p, v, height=0.8, color="tab:red", alpha=0.6, edgecolor="black", linewidth=0.5)
    ax.text(v + 0.5, p, f"{int(v)}@{int(p)}", va="center", ha="left", fontsize=9, color="tab:red")

# Wall Mid line
ax.axhline(wm, color="orange", linewidth=2, linestyle="--", label=f"Wall Mid = {wm:.1f}")

# My simulated quotes
my_bid_price = wm + MY_BID_OFFSET
my_ask_price = wm + MY_ASK_OFFSET
ax.barh(my_bid_price, -MY_QUOTE_SIZE, height=0.5, color="lime", alpha=0.8,
        edgecolor="black", linewidth=1.5, hatch="//")
ax.text(-MY_QUOTE_SIZE - 0.5, my_bid_price,
        f"MY BID: {MY_QUOTE_SIZE}@{my_bid_price:.0f} (edge={abs(MY_BID_OFFSET)})",
        va="center", ha="right", fontsize=10, fontweight="bold", color="green")

ax.barh(my_ask_price, MY_QUOTE_SIZE, height=0.5, color="lime", alpha=0.8,
        edgecolor="black", linewidth=1.5, hatch="//")
ax.text(MY_QUOTE_SIZE + 0.5, my_ask_price,
        f"MY ASK: {MY_QUOTE_SIZE}@{my_ask_price:.0f} (edge={abs(MY_ASK_OFFSET)})",
        va="center", ha="left", fontsize=10, fontweight="bold", color="green")

# Spread annotation
if bid_levels and ask_levels:
    best_bid = max(p for p, v in bid_levels)
    best_ask = min(p for p, v in ask_levels)
    ax.annotate("", xy=(0, best_ask), xytext=(0, best_bid),
                arrowprops=dict(arrowstyle="<->", color="purple", lw=2))
    ax.text(0.5, (best_bid + best_ask) / 2,
            f"Spread = {best_ask - best_bid:.0f}",
            fontsize=10, color="purple", fontweight="bold")

ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Volume (← Bids | Asks →)")
ax.set_ylabel("Price")
ax.set_title(f"Orderbook Depth Snapshot — {PRODUCT} @ t={int(snap['timestamp'])}", fontweight="bold", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

---
## 9b: Fill Probability vs Edge Curve

**The single most important plot for tuning aggressiveness.**

For each possible edge (1, 2, 3, ... ticks from fair value), computes:
- **Fill rate**: What % of timesteps had a trade that crossed your price?
- **Average edge per fill**: When filled, how much edge did you capture?
- **Expected value per timestep**: fill_rate x edge (the product)

The optimal aggressiveness is where **expected value peaks**.
Tighter quotes = more fills but less edge per fill.
Wider quotes = fewer fills but more edge per fill.

In [ ]:
# ============================================================
# 9b: FILL PROBABILITY vs EDGE CURVE
# ============================================================

wm_series = prices.set_index("timestamp")["wall_mid"]

results_by_edge = []
for edge in EDGE_TEST_RANGE:
    # For each timestep, check if any trade crossed our quote level
    bid_fills = 0
    ask_fills = 0
    total_steps = 0

    for ts in prices["timestamp"].unique():
        wm_val = wm_series.get(ts, np.nan)
        if np.isnan(wm_val):
            continue
        total_steps += 1

        ts_trades = trades[trades["timestamp"] == ts]
        for _, trade in ts_trades.iterrows():
            if trade["price"] >= wm_val + edge:
                ask_fills += 1
            if trade["price"] <= wm_val - edge:
                bid_fills += 1

    total_fills = bid_fills + ask_fills
    fill_rate = total_fills / total_steps if total_steps > 0 else 0
    ev_per_step = fill_rate * edge

    results_by_edge.append({
        "edge": edge,
        "bid_fills": bid_fills,
        "ask_fills": ask_fills,
        "total_fills": total_fills,
        "fill_rate": fill_rate,
        "ev_per_step": ev_per_step,
        "total_steps": total_steps,
    })

edge_df = pd.DataFrame(results_by_edge)

fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH, FIG_HEIGHT))

# Fill rate
axes[0].bar(edge_df["edge"], edge_df["fill_rate"] * 100, color="tab:blue", alpha=0.7)
axes[0].set_xlabel("Edge (ticks from fair)")
axes[0].set_ylabel("Fill Rate (%)")
axes[0].set_title("Fill Rate vs Edge", fontweight="bold")
axes[0].grid(True, alpha=0.3, axis="y")

# Edge per fill (constant in this simple model, but shows the line)
axes[1].bar(edge_df["edge"], edge_df["edge"], color="tab:orange", alpha=0.7)
axes[1].set_xlabel("Edge (ticks from fair)")
axes[1].set_ylabel("Edge per Fill")
axes[1].set_title("Edge per Fill", fontweight="bold")
axes[1].grid(True, alpha=0.3, axis="y")

# Expected value = fill_rate * edge
axes[2].bar(edge_df["edge"], edge_df["ev_per_step"], color="tab:green", alpha=0.7)
best_edge = edge_df.loc[edge_df["ev_per_step"].idxmax(), "edge"]
axes[2].axvline(best_edge, color="red", linestyle="--", linewidth=2, label=f"Optimal edge = {best_edge}")
axes[2].set_xlabel("Edge (ticks from fair)")
axes[2].set_ylabel("Expected Value per Step")
axes[2].set_title("Expected Value vs Edge", fontweight="bold")
axes[2].legend(fontsize=10)
axes[2].grid(True, alpha=0.3, axis="y")

fig.suptitle(f"Fill Probability vs Edge — {PRODUCT} (Day {DAY})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Buy vs Sell side breakdown
print("\nDetailed edge analysis:")
print(edge_df[["edge", "bid_fills", "ask_fills", "total_fills", "fill_rate", "ev_per_step"]].to_string(
    index=False, float_format="{:.4f}".format
))
print(f"\n>>> Optimal edge: {best_edge} (EV/step = {edge_df.loc[edge_df['ev_per_step'].idxmax(), 'ev_per_step']:.4f})")

---
## 9c: Inventory Trajectory Simulator

Simulates how your position (inventory) evolves over a full day given
your quoting parameters. Shows:
- Position over time
- When you hit position limits (= dead time, no more quoting)
- The effect of different flattening rules

**Key insight from top teams:** Time spent at position limits is wasted
opportunity. If your inventory management is too slow, you lose PnL.

In [ ]:
# ============================================================
# 9c: INVENTORY TRAJECTORY SIMULATOR
# ============================================================

def simulate_inventory(prices_df, trades_df, edge, max_pos, flatten_thresh, flatten_edge=0):
    """
    Simulates inventory evolution for a market-making strategy.

    Returns dict with position_series, pnl_series, time_at_limit,
    flatten_events, total_pnl.
    """
    wm = prices_df["wall_mid"].values
    timestamps = prices_df["timestamp"].values
    position = 0
    cash = 0.0
    positions = []
    pnls = []
    flatten_events = []
    time_at_limit = 0

    for i in range(len(wm)):
        fair = wm[i]
        if np.isnan(fair):
            positions.append(position)
            pnls.append(cash)
            continue

        # Check if at position limit
        if abs(position) >= max_pos:
            time_at_limit += 1

        # Passive fills from trades
        ts_trades = trades_df[trades_df["timestamp"] == timestamps[i]]
        for _, trade in ts_trades.iterrows():
            if trade["price"] >= fair + edge and position > -max_pos:
                qty = min(int(trade["quantity"]), max_pos + position)
                if qty > 0:
                    cash += (fair + edge) * qty
                    position -= qty
            elif trade["price"] <= fair - edge and position < max_pos:
                qty = min(int(trade["quantity"]), max_pos - position)
                if qty > 0:
                    cash -= (fair - edge) * qty
                    position += qty

        # Flatten if over threshold
        if abs(position) >= flatten_thresh:
            flatten_price = fair + flatten_edge * np.sign(position)
            cash += position * flatten_price
            flatten_events.append(timestamps[i])
            position = 0

        positions.append(position)
        pnls.append(cash + position * fair)

    last_fair = wm[~np.isnan(wm)][-1] if any(~np.isnan(wm)) else 0
    total_pnl = cash + position * last_fair

    return {
        "positions": np.array(positions),
        "pnls": np.array(pnls),
        "flatten_events": flatten_events,
        "time_at_limit": time_at_limit,
        "total_pnl": total_pnl,
        "timestamps": timestamps,
    }

sim = simulate_inventory(prices, trades, SIM_EDGE, SIM_MAX_POSITION, SIM_FLATTEN_THRESHOLD, SIM_FLATTEN_EDGE)

fig, axes = plt.subplots(3, 1, figsize=(FIG_WIDTH, FIG_HEIGHT * 2), sharex=True)

# Position over time
axes[0].plot(sim["timestamps"], sim["positions"], linewidth=0.8, color="tab:blue")
axes[0].axhline(SIM_MAX_POSITION, color="red", linestyle="--", alpha=0.6, label=f"±{SIM_MAX_POSITION} limit")
axes[0].axhline(-SIM_MAX_POSITION, color="red", linestyle="--", alpha=0.6)
axes[0].axhline(SIM_FLATTEN_THRESHOLD, color="orange", linestyle=":", alpha=0.6, label=f"±{SIM_FLATTEN_THRESHOLD} flatten")
axes[0].axhline(-SIM_FLATTEN_THRESHOLD, color="orange", linestyle=":", alpha=0.6)
axes[0].axhline(0, color="black", linewidth=0.5)
for fe in sim["flatten_events"]:
    axes[0].axvline(fe, color="green", alpha=0.3, linewidth=0.5)
axes[0].set_ylabel("Position")
axes[0].set_title(f"Inventory Trajectory (edge={SIM_EDGE}, flatten@{SIM_FLATTEN_THRESHOLD})", fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# PnL curve
axes[1].plot(sim["timestamps"], sim["pnls"], linewidth=0.8, color="tab:green")
axes[1].set_ylabel("Cumulative PnL")
axes[1].set_title("PnL Over Time", fontweight="bold")
axes[1].grid(True, alpha=0.3)

# Position histogram
axes[2].hist(sim["positions"], bins=50, color="tab:purple", alpha=0.6)
axes[2].set_xlabel("Position")
axes[2].set_ylabel("Count")
axes[2].set_title("Position Distribution", fontweight="bold")
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

print(f"\nSimulation results:")
print(f"  Total PnL:        {sim['total_pnl']:.0f}")
print(f"  Flatten events:   {len(sim['flatten_events'])}")
print(f"  Time at limit:    {sim['time_at_limit']} steps ({sim['time_at_limit']/len(sim['timestamps'])*100:.1f}%)")
print(f"  Avg |position|:   {np.mean(np.abs(sim['positions'])):.1f}")
print(f"  Max |position|:   {np.max(np.abs(sim['positions']))}")

---
## 9d: Quote Competitiveness Analysis

At each timestep, checks whether your simulated bid/ask would be the
**best in book** or if existing liquidity is ahead of you.

- If your bid is above the current best bid, you're overbidding (good for fills)
- If your bid is below the current best bid, you're behind (fewer fills)
- Same logic applies to the ask side

In [ ]:
# ============================================================
# 9d: QUOTE COMPETITIVENESS ANALYSIS
# ============================================================

test_edge = SIM_EDGE
wm_arr = prices["wall_mid"].values
best_bid = prices["bid_price_1"].values
best_ask = prices["ask_price_1"].values

my_bid = wm_arr - test_edge
my_ask = wm_arr + test_edge

# How does my bid compare to existing best bid?
bid_vs_best = my_bid - best_bid  # Positive = I'm overbidding (good)
ask_vs_best = best_ask - my_ask  # Positive = I'm undercutting (good)

fig, axes = plt.subplots(1, 2, figsize=(FIG_WIDTH, FIG_HEIGHT))

# Bid competitiveness
valid_bid = ~np.isnan(bid_vs_best)
axes[0].hist(bid_vs_best[valid_bid], bins=30, color="tab:blue", alpha=0.7)
axes[0].axvline(0, color="red", linewidth=2, linestyle="--", label="Tied with best bid")
pct_overbid = (bid_vs_best[valid_bid] > 0).mean() * 100
axes[0].set_xlabel("My Bid − Best Bid (positive = overbidding)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Bid Competitiveness (edge={test_edge})\n{pct_overbid:.0f}% of time I overbid", fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis="y")

# Ask competitiveness
valid_ask = ~np.isnan(ask_vs_best)
axes[1].hist(ask_vs_best[valid_ask], bins=30, color="tab:red", alpha=0.7)
axes[1].axvline(0, color="red", linewidth=2, linestyle="--", label="Tied with best ask")
pct_undercut = (ask_vs_best[valid_ask] > 0).mean() * 100
axes[1].set_xlabel("Best Ask − My Ask (positive = undercutting)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Ask Competitiveness (edge={test_edge})\n{pct_undercut:.0f}% of time I undercut", fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

---
## 9e: Aggressiveness Comparison Table

Runs multiple aggressiveness configurations side by side.
Each row shows: edge, fill rate, PnL, inventory stats.

**How to pick your setting:**
- Choose the setting with the best PnL that ALSO has low time-at-limit
- If two settings have similar PnL, prefer the one with lower variance
- Cross-check against the other days to ensure stability

In [ ]:
# ============================================================
# 9e: AGGRESSIVENESS COMPARISON TABLE
# ============================================================

comparison_rows = []
for config in COMPARISON_CONFIGS:
    sim = simulate_inventory(
        prices, trades,
        config["edge"], config["max_pos"], config["flatten"],
    )
    comparison_rows.append({
        "Setting": config["label"],
        "Edge": config["edge"],
        "Max Pos": config["max_pos"],
        "Flatten@": config["flatten"],
        "Total PnL": sim["total_pnl"],
        "Flatten Events": len(sim["flatten_events"]),
        "Avg |Pos|": np.mean(np.abs(sim["positions"])),
        "Max |Pos|": np.max(np.abs(sim["positions"])),
        "Time@Limit %": sim["time_at_limit"] / len(sim["timestamps"]) * 100,
    })

comp_df = pd.DataFrame(comparison_rows)
print(f"Aggressiveness Comparison — {PRODUCT} (Day {DAY})\n")
print(comp_df.to_string(index=False, float_format="{:.1f}".format))

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(FIG_WIDTH, FIG_HEIGHT))

labels = comp_df["Setting"]
x = range(len(labels))

axes[0].bar(x, comp_df["Total PnL"], color="tab:green", alpha=0.7)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, rotation=20)
axes[0].set_ylabel("Total PnL")
axes[0].set_title("PnL by Setting", fontweight="bold")
axes[0].grid(True, alpha=0.3, axis="y")

axes[1].bar(x, comp_df["Flatten Events"], color="tab:orange", alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=20)
axes[1].set_ylabel("Flatten Events")
axes[1].set_title("Flatten Frequency", fontweight="bold")
axes[1].grid(True, alpha=0.3, axis="y")

axes[2].bar(x, comp_df["Time@Limit %"], color="tab:red", alpha=0.7)
axes[2].set_xticks(x)
axes[2].set_xticklabels(labels, rotation=20)
axes[2].set_ylabel("Time at Limit (%)")
axes[2].set_title("Wasted Time at Limit", fontweight="bold")
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CROSS-DAY STABILITY CHECK
# ============================================================
# Run the comparison across all 3 days to ensure your chosen
# parameters are robust, not overfit to a single day.

print(f"Cross-day stability check for {PRODUCT}:\n")
for config in COMPARISON_CONFIGS:
    pnls = []
    for d in AVAILABLE_DAYS:
        p = compute_wall_mid(load_prices(day=d, product=PRODUCT))
        t = load_trades(day=d, product=PRODUCT)
        s = simulate_inventory(p, t, config["edge"], config["max_pos"], config["flatten"])
        pnls.append(s["total_pnl"])
    print(f"  {config['label']} (edge={config['edge']}): "
          f"Day -2={pnls[0]:.0f}, Day -1={pnls[1]:.0f}, Day 0={pnls[2]:.0f}, "
          f"Avg={np.mean(pnls):.0f}, Std={np.std(pnls):.0f}")